# Instinct:从零训练 64M LLM —— 交互式导览

本 notebook 带你**逐步看完**这个仓库的一整条链路:**模型 → 数据 → 训练 → 推理**。
所有代码都直接复用仓库真实模块(`model/`、`trainer/`、`dataset/`),没有隐藏的魔改,
每段都可以独立改参数重跑。项目哲学见 [AGENTS.md](./AGENTS.md):**每一行代码都可读、可改、可复现**。

## 0. 运行前须知

| 项目 | 说明 |
|------|------|
| 前置依赖 | `pip install -r requirements.txt`(PyTorch、transformers、datasets 等,需联网) |
| 运行方式 | 菜单 **Cell → Run All**(或逐个 Shift+Enter),总耗时约 1–3 分钟 |
| 硬件 | 有 NVIDIA GPU 用 GPU(快);没有则自动落回 CPU(演示配置很小,也跑得动) |
| 位置 | 必须把本 notebook 放在**仓库根目录**(与 `AGENTS.md` 同级)运行 |
| Windows 注意 | 本仓库先 `import datasets` 再 `import torch`,规避 pyarrow/torch DLL 冲突(见 AGENTS.md);第 1 个 cell 已按此顺序 |

## 目录

1. **环境与模型** —— 版本检查、`InstinctConfig` 关键默认值、四种架构参数量、一次前向
2. **从零训练一个小模型** —— 真实数据预览 → 复用 `trainer_cli`/`trainer_utils` 构建配置 → 60 步训练循环
3. **推理** —— 刚训练的小模型生成一句;再加载仓库真实 64M 预训练权重来一次“正式”对话
4. **走向生产** —— 完整训练 / 模型转换 / 部署的命令速查

> 提示:第 2 节训练的是 128 维、2 层的“玩具”模型(秒级跑完),它只证明**管线**;
> 真正的效果请跳第 3 节用仓库里现成的 64M 权重体验。


In [ ]:
# ① 环境准备:仓库根目录 + 版本信息
# Windows:先 import datasets 再 import torch(规避 pyarrow/torch DLL 冲突,见 AGENTS.md)
import os, sys
import datasets  # noqa: F401
import torch

ROOT = os.path.abspath(os.getcwd())
assert os.path.exists(os.path.join(ROOT, "AGENTS.md")), "请把本 notebook 放在仓库根目录运行"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("PyTorch :", torch.__version__)
print("datasets :", datasets.__version__)
print("CUDA 可用:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(无,将用 CPU)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("计算设备:", DEVICE)


## 第 1 节 模型

`model/model_instinct.py` 实现了 Qwen3 生态风格的 Dense Transformer:
**Pre-Norm RMSNorm + RoPE(θ=1e6)+ GQA(8 头 / 4 KV 头)+ SwiGLU**,词表 6400、最大上下文 32768。
同一套配置类还派生出三种变体(见左下表格),训练脚本用 `--use_moe / --use_looped` 一行切换。


In [ ]:
# ② InstinctConfig 关键默认值 —— 模型“图纸”
from model.model_instinct import InstinctConfig
cfg = InstinctConfig()
for k in ["hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads",
          "vocab_size", "max_position_embeddings", "rope_theta", "tie_word_embeddings", "use_moe"]:
    print(f"{k:22s} = {getattr(cfg, k)}")


In [ ]:
# ③ 四种架构参数量对比(统一用 128 维 / 2 层的“玩具”配置,秒级构建)
from model.model_instinct import InstinctConfig, InstinctForCausalLM
from model.model_instinct_loop import InstinctConfig as LoopConfig, InstinctForCausalLM as LoopLM
from model.model_instinct_linear import InstinctConfig as LinConfig, InstinctForCausalLM as LinLM

TINY_DENSE = InstinctForCausalLM(InstinctConfig(hidden_size=128, num_hidden_layers=2))
TINY_MOE   = InstinctForCausalLM(InstinctConfig(hidden_size=128, num_hidden_layers=2,
                                                use_moe=True, num_experts=4, num_experts_per_tok=1))
TINY_LOOP  = LoopLM(LoopConfig(hidden_size=128, num_hidden_layers=2, loop_iters=4,
                               prelude_layers=1, coda_layers=1))
TINY_LIN   = LinLM(LinConfig(hidden_size=128, num_hidden_layers=2))

def total_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

for name, m in [("Dense(标准)", TINY_DENSE), ("MoE(4专家,Top1)", TINY_MOE),
                ("Loop(共享块×4)", TINY_LOOP), ("Linear(GatedDeltaNet)", TINY_LIN)]:
    print(f"{name:22s} 总参数量 = {total_params(m):8.3f} M")

# MoE 的“激活参数”:每个 token 只走 1 个专家,所以推理/训练时实际算力远小于总参数量
exp = sum(p.numel() for n, p in TINY_MOE.named_parameters() if "experts.0." in n) / 1e6
base = total_params(TINY_MOE) - 4 * exp
print("\nMoE 激活(每个 token): base %.3f M + 1 专家 %.3f M = %.3f M" % (base, exp, base + exp))


In [ ]:
# ④ 一次前向:Dense 模型输入 token → logits + 交叉熵损失
torch.manual_seed(0)
x = torch.randint(0, cfg.vocab_size, (2, 16))
out = TINY_DENSE(x, labels=x)
print("logits 形状:", tuple(out.logits.shape))          # (batch, seq, vocab)
print("loss      :", round(float(out.loss.item()), 4))


In [ ]:
# ⑤ MoE 的 router aux_loss:训练脚本里 loss = 主损失 + aux_loss(负载均衡正则)
out_m = TINY_MOE(x, labels=x)
print("主损失   :", round(float(out_m.loss.item()), 4))
print("aux_loss :", round(float(out_m.aux_loss.item()), 4))
print("合计     :", round(float((out_m.loss + out_m.aux_loss).item()), 4))


## 第 2 节 从零训练一个小模型

训练脚本(`trainer/train_pretrain.py`)做的事,这里**原样复刻但不藏起来**:
1. `build_trainer_parser` 解析命令行参数(仓库统一在 `trainer/trainer_cli.py`)
2. `config_from_args` 把参数变成模型配置
3. `PretrainDataset` 装载 `{"text": ...}` 语料(每行一个 JSON)
4. 训练循环:余弦退火学习率 + 混合精度反向 + 梯度累积裁剪更新

下面四步完全复用仓库真实函数 —— 你改几个数字就能变成自己的实验。


In [ ]:
# ⑥ 数据:先看真实预训练数据格式,再取前 8 条做成小文件(完整 mini 文件 1.2GB,演示不必全读)
import json, itertools
RAW = os.path.join("dataset", "pretrain_t2t_mini.jsonl")
with open(RAW, encoding="utf-8") as f:
    first_rows = [json.loads(line) for line in itertools.islice(f, 8)]
for i, r in enumerate(first_rows[:3]):
    print(f"[{i}] {r['text'][:60]}…")
demo_data = os.path.join("out", "demo_pretrain.jsonl")
os.makedirs("out", exist_ok=True)
with open(demo_data, "w", encoding="utf-8") as f:
    for r in first_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"\n已生成演示语料(前 {len(first_rows)} 条):", demo_data)


In [ ]:
# ⑦ 用真实 trainer 管线构建模型配置(不手写 argparse,完全复用仓库代码)
from trainer.trainer_cli import build_trainer_parser
from trainer.trainer_utils import config_from_args

parser = build_trainer_parser("Notebook 演示")
args = parser.parse_args([])                        # 全部默认值,与 train_pretrain.py 相同
lm_config = config_from_args(args, hidden_size=128, num_hidden_layers=2)
print("learning_rate =", args.learning_rate, "| grad_clip =", args.grad_clip)
print("use_moe =", lm_config.use_moe, "| vocab_size =", lm_config.vocab_size)
print("max_seq_len =", args.max_seq_len)


In [ ]:
# ⑧ 数据装载(PretrainDataset,与训练脚本同款)+ 构建小模型 + AdamW 优化器
from torch.utils.data import DataLoader
from dataset.lm_dataset import PretrainDataset
from trainer.trainer_utils import build_optimizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./model")
ds = PretrainDataset(demo_data, tokenizer, max_length=64)
loader = DataLoader(ds, batch_size=4, shuffle=True)
print("训练样本数:", len(ds), "| 每条长度:", 64, "tokens")

model_small = InstinctForCausalLM(lm_config).to(DEVICE)
optimizer = build_optimizer(model_small.parameters(), args.learning_rate, args.optimizer)
print("小型模型参数量: %.2f M" % (sum(p.numel() for p in model_small.parameters()) / 1e6))


In [ ]:
# ⑨ 训练循环(60 步):set_cosine_lr 余弦退火 + GradScaler 反向 + step_with_scaler 更新
# 这三个辅助函数来自 trainer/trainer_cli.py —— 与 8 个训练脚本共用同一套,不重复造轮子
from trainer.trainer_cli import set_cosine_lr, step_with_scaler

TOTAL_STEPS = 60
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
losses = []
model_small.train()
for step, (input_ids, labels) in enumerate(itertools.cycle(loader)):  # 数据集只有 8 条,循环复用直到 60 步
    if step >= TOTAL_STEPS:
        break
    input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)
    set_cosine_lr(optimizer, 0, step, TOTAL_STEPS, args)              # 余弦退火 LR
    with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):         # 混合精度前向
        res = model_small(input_ids, labels=labels)
    loss = res.loss + res.aux_loss
    scaler.scale(loss).backward()                                    # 缩放反向
    step_with_scaler(scaler, optimizer, model_small.parameters(), args.grad_clip)  # 裁剪+更新+清零
    losses.append(float(loss.item()))
    if step % 20 == 0:
        print(f"  step {step:3d}  loss {float(loss.item()):.4f}")

print(f"训练完成:{len(losses)} 步")
print("loss 首 / 中 / 尾:", [round(v, 3) for v in (losses[0], losses[len(losses) // 2], losses[-1])])
print("loss 降幅      :", round(losses[0] - losses[-1], 3))


## 第 3 节 推理

两个推理体验:
1. 用刚训练(仅 60 步)的**小模型**生成 —— 它只学到了皮毛,重点是看**管线从训练到推理是连通的**
2. 加载仓库里现成的 **64M 预训练权重**(768 维、8 层,`out/pretrain_20260824_003848_768.pth`)来一次正式对话


In [ ]:
# ⑩ 小模型推理(60 步训练后,效果有限,但“训练→推理”闭环成立)
model_small.eval()
prompt = "人工智能正在"
ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
inputs = torch.tensor([ids], device=DEVICE)
with torch.no_grad():
    gen = model_small.generate(inputs, max_new_tokens=24, do_sample=True, temperature=0.85)
print("输入:", prompt)
print("输出:", tokenizer.decode(gen[0], skip_special_tokens=True))
print("\n说明:小模型参数量不足、训练步数极少,生成的文字仅证明管线连通。")


In [ ]:
# ⑪ 加载仓库真实 64M 预训练权重做推理(768 维 / 8 层),自动挑选 out/ 下最新 Dense 权重
# init_model 就是训练脚本里加载权重的同一个函数:构造模型 + load_state_dict(strict=False)
import glob, time as _t
from trainer.trainer_utils import init_model

cands = [p for p in glob.glob(os.path.join("out", "pretrain*.pth")) if "_moe" not in p and "_linear" not in p]
cands.sort(key=os.path.getmtime, reverse=True)          # 最新生成者优先
if not cands:
    print("未找到 out/pretrain*_768.pth,跳过本 cell 的推理演示。")
else:
    weight = os.path.basename(cands[0])[:-len("_768.pth")]
    print("使用权重:", os.path.basename(cands[0]))
    big_cfg = InstinctConfig()                          # 默认 768 配置,与权重匹配
    t0 = _t.time()
    model_big, tokenizer_big = init_model(big_cfg, from_weight=weight, device=DEVICE)
    print(f"权重加载耗时 {_t.time() - t0:.1f}s")
    model_big.eval()

    prompt2 = "你好,请介绍一下你自己。"
    ids2 = tokenizer_big(prompt2, add_special_tokens=False)["input_ids"]
    inputs2 = torch.tensor([ids2], device=DEVICE)
    t0 = _t.time()
    with torch.no_grad():
        gen2 = model_big.generate(inputs2, max_new_tokens=32, do_sample=True, temperature=0.85,
                                  top_p=0.85, top_k=50)
    print(f"生成耗时 {_t.time() - t0:.1f}s(CPU 每 token 约 1s,GPU 瞬时)")
    print("Prompt:", prompt2)
    print("回复  :", tokenizer_big.decode(gen2[0], skip_special_tokens=True))


## 第 4 节 走向生产(命令速查,不在本 notebook 里执行)

### 完整训练(仓库根目录运行)
```bash
# 预训练(第 1 阶段,必须)
python trainer/train_pretrain.py
# 指令微调(第 2 阶段,必须,基于预训练权重)
python trainer/train_full_sft.py --from_weight pretrain
# 可选:LoRA / DPO / PPO / GRPO / Agentic RL / 蒸馏
python trainer/train_lora.py --from_weight full_sft
python trainer/train_dpo.py --from_weight full_sft
# 断点续训
python trainer/train_pretrain.py --from_resume 1
# 多卡 DDP
torchrun --nproc_per_node N trainer/train_pretrain.py
```

### 权重转换 + 部署
```bash
cd scripts
python convert_model.py                      # torch .pth <-> HuggingFace 格式
streamlit run web_demo.py                    # WebUI(模型目录需先复制到 scripts/ 下)
python serve_openai_api.py                   # OpenAI 兼容 API(localhost:8998)
```

### 第三方生态
转换后可直接用 `llama.cpp`(GGUF)/ `vllm` / `ollama` 加载(权重对齐 Qwen3 生态)。

> 本仓库训练日志默认走 SwanLab(WandB 兼容),国内直连稳定;需要时加 `--use_wandb`。


## 小结

你已经看完:**配置 → 四种架构 → 前向 → 真实数据 → 复用 trainer 管线的训练循环 → 推理 → 部署出口**。

动手建议:
1. 把第 2 节改成你自己的实验(换 `learning_rate`、`TOTAL_STEPS`、`hidden_size`,观察 loss 曲线变化)
2. 换成 `sft_t2t_mini.jsonl` + `SFTDataset`,体验指令微调
3. 跑完整训练脚本前,先读 `trainer/train_pretrain.py` —— 你会发现里面的循环和第 ⑨ 步几乎一模一样

一路看下来你会发现:仓库没有魔法,`trainer/trainer_cli.py` 里那几个辅助函数就是全部训练脚本的公共骨架。
